# Hands-on Decision Tree Classification with Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mojiahui777/ba_course/blob/main/decision_tree_credit_risk_demo.ipynb)

> **Goal:** Learn how to train, interpret, and evaluate a decision tree.
>
> Run the `Credit.csv` example first. Then upload another CSV, change the settings, and run the model again.

## Learning objectives

- train an entropy-based decision tree;
- interpret nodes, branches, leaves, and rules;
- evaluate predictions with accuracy and a confusion matrix; and
- test the same model with another classification dataset.


## 1. Define the business problem

Before coding, write down:

- **Observation:** What does one row represent?
- **Target:** What category are you trying to predict?
- **Predictors:** What information is available before the prediction?
- **Decision:** How would someone use the prediction?
- **Error costs:** Are the two types of classification error equally serious?

**Example:** In `Credit.csv`, one row represents an applicant and `CREDIT` is the recorded class. The file does not verify whether `0` or `1` means Good, so keep the numeric labels unless a matching data dictionary supplies that meaning.

## 2. Import the Python libraries

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

## 3. Load the data

- Use the default URL for `Credit.csv`.
- Change `UPLOAD_FILE` to `True` to upload another CSV in Colab.
- Or replace `DATA_URL` with another direct CSV link.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mojiahui777/ba_course/main/Credit.csv"
UPLOAD_FILE = False

if UPLOAD_FILE:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    df = pd.read_csv(file_name)
else:
    df = pd.read_csv(DATA_URL)

df.head()

## 4. Choose the columns and model settings

When using another dataset, list only the predictors you want the model to use. Columns not listed below are ignored.

In [ ]:
# Edit these values for another dataset
TARGET_COLUMN = "CREDIT"

# Only columns listed here will be used by the model
NUMERICAL_COLUMNS = ["DURATION", "AGE", "Amount"]
CATEGORICAL_COLUMNS = [
    "CHK_ACCT", "JOB", "NUM_DEPENDENTS", "TELEPHONE", "FOREIGN"
]

TEST_SIZE = 0.20
MAX_DEPTH = 3
RANDOM_STATE = 42

## 5. Preview the data

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())
display(df[TARGET_COLUMN].value_counts().rename("count").to_frame())

## 6. Prepare X and y

- `X` contains only the selected numerical and categorical predictors.
- `y` contains the target.
- Columns not selected in Section 4 are not used.
- `get_dummies` converts categorical predictors into 0/1 columns.

In [ ]:
SELECTED_COLUMNS = NUMERICAL_COLUMNS + CATEGORICAL_COLUMNS
X = df[SELECTED_COLUMNS].copy()
y = df[TARGET_COLUMN]

X = pd.get_dummies(X, columns=CATEGORICAL_COLUMNS, dtype=int)
display(X.head())

## 7. Create an 80/20 train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

## 8. Train the decision tree

- `criterion="entropy"` selects splits using information gain.
- `MAX_DEPTH` controls the size of the tree.

In [ ]:
tree_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE,
)

tree_model.fit(X_train, y_train)

## 9. Evaluate holdout predictions

- **Accuracy** is the overall correct share.
- **Balanced accuracy** gives equal weight to recall for each class.
- **Precision, recall, and F1** show class-specific performance.

Always compare the tree with the majority baseline.

In [ ]:
y_pred = tree_model.predict(X_test)
baseline_accuracy = y_train.value_counts(normalize=True).max()

print(f"Test accuracy:          {accuracy_score(y_test, y_pred):.3f}")
print(f"Balanced accuracy:      {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"Majority baseline:      {baseline_accuracy:.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 10. Read the confusion matrix

Rows are actual classes and columns are predicted classes. Off-diagonal cells are errors. Translate those errors into business meaning using your dataset's verified class definitions.

In [ ]:
class_labels = list(tree_model.classes_)
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=class_labels,
    cmap="Reds",
    colorbar=False,
)
plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.tight_layout()
plt.show()

## 11. Visualize and read the tree

Start at the root. Move left when a condition is true and right when it is false. Each node reports entropy, sample count, class counts, and the predicted class.

In [ ]:
feature_names = list(X.columns)

plt.figure(figsize=(18, 9))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=[str(value) for value in tree_model.classes_],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title(f"Decision Tree (Maximum Depth {MAX_DEPTH})")
plt.tight_layout()
plt.show()

## 12. Translate the tree into text rules

In [ ]:
print(export_text(tree_model, feature_names=list(feature_names)))

## 13. Explore feature importance

Importance summarizes impurity reduction in this fitted tree. It is not proof of causality, fairness, or stability.

In [ ]:
importance = pd.Series(
    tree_model.feature_importances_, index=feature_names
)
importance = importance[importance > 0].sort_values()

importance.plot(kind="barh", figsize=(8, 4), color="#CC0000")
plt.xlabel("Feature importance")
plt.title("Features Used by the Tree")
plt.tight_layout()
plt.show()

## 14. Practice with another dataset

For your new dataset, submit:

1. the edited settings cell;
2. the fitted decision tree;
3. two decision rules;
4. accuracy, baseline, and confusion matrix interpretation; and
5. two model limitations.

**Takeaway:** Explain how the tree makes predictions and whether it performs well on unseen data.